# DSC 2026 — Task 1 (LegalIR)

## Hướng dẫn sử dụng

1. **Bật GPU T4 x2**: Notebook (phải) → Settings → Accelerator → `GPU T4 x2`.
2. **Bật Internet**: Notebook (phải) → Settings → Internet → `On`.
3. **Đưa dữ liệu vào**: "Add Data" → tạo Dataset chứa `context_*.json`, `train.json`, `public-official.json` → attach vào notebook. Sửa `KAGGLE_INPUT_DATA_DIR` ở phần Cấu hình cho khớp tên dataset.
4. **Tái sử dụng index ở lần chạy sau**: sau khi chạy xong 1 lần, vào tab Output → New Dataset từ thư mục `index_cache/` → attach ở lần sau → điền vào `CACHED_INDEX_INPUT_DIR` ở phần Cấu hình. Notebook sẽ nạp lại thay vì build từ đầu.
5. **Chạy bị ngắt giữa chừng**: chạy lại đúng cell ở phần "Chạy pipeline & Lưu kết quả" — tự đọc file checkpoint, chỉ xử lý tiếp các câu hỏi còn thiếu.
6. **Muốn chạy lại từ đầu**: xóa `outputs/submission_checkpoint.json` trước khi chạy lại.
7. **Chiến lược**: theo xác nhận của BTC, Recall là tiêu chí xếp hạng chính (Precision chỉ để phân định khi Recall bằng nhau) — pipeline luôn trả về đúng 5 document_id cho mỗi câu hỏi thay vì số lượng thay đổi.
8. **So với bản gốc**: notebook này chỉ thêm 2 thay đổi cốt lõi — (a) chunking sub-split các "Điều" quá dài để không bị model cắt cụt khi encode, (b) tuỳ chọn fusion **RRF** bên cạnh cộng điểm có trọng số cũ (`FUSION_METHOD` trong phần Cấu hình). Mọi phần khác giữ nguyên như bản gốc.


## Cài đặt thư viện

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


In [2]:
!pip install -q rank_bm25 pyvi sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 47.2 MB/s eta 0:00:00


In [3]:
import re
import gc
import json
import glob
import pickle
import random
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import torch
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from pyvi import ViTokenizer
from sentence_transformers import SentenceTransformer, CrossEncoder, util

print("CUDA available:", torch.cuda.is_available())
print("Số GPU:", torch.cuda.device_count())


CUDA available: True
Số GPU: 2


## Cấu hình

In [4]:
KAGGLE_INPUT_DATA_DIR = "/kaggle/input/datasets/remiiyoo/legalir-data"
CONTEXT_DIR = os.path.join(KAGGLE_INPUT_DATA_DIR, "selected-contexts/selected-contexts")
TRAIN_FILE = os.path.join(KAGGLE_INPUT_DATA_DIR, "train.json")
PUBLIC_TEST_FILE = os.path.join(KAGGLE_INPUT_DATA_DIR, "public-official.json")

CACHED_INDEX_INPUT_DIR = None
# CACHED_INDEX_INPUT_DIR = "/kaggle/input/legalir-index-cache"

WORKING_DIR = "/kaggle/working"
INDEX_DIR = os.path.join(WORKING_DIR, "index_cache")
OUTPUT_DIR = os.path.join(WORKING_DIR, "outputs")
SUBMISSION_FILE = os.path.join(OUTPUT_DIR, "submission.json")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "submission_checkpoint.json")

os.makedirs(INDEX_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


def resolve_index_read_dir():
    if CACHED_INDEX_INPUT_DIR is not None:
        cached_file = os.path.join(CACHED_INDEX_INPUT_DIR, "dense_embeddings.pt")
        if os.path.exists(cached_file):
            return CACHED_INDEX_INPUT_DIR
    return INDEX_DIR


DENSE_MODEL_NAME = "AITeamVN/Vietnamese_Embedding"
RERANKER_MODEL_NAME = "AITeamVN/Vietnamese_Reranker"
MAX_SEQ_LENGTH = 1024
RERANKER_MAX_LENGTH = 1024

GPU_DEVICES = []
for i in range(torch.cuda.device_count()):
    GPU_DEVICES.append(f"cuda:{i}")
if len(GPU_DEVICES) == 0:
    GPU_DEVICES = ["cpu"]
PRIMARY_DEVICE = GPU_DEVICES[0]

MAX_WORDS_PER_CHUNK = 200
CHUNK_OVERLAP = 30
INCLUDE_APPENDIX = True

FUSION_METHOD = "rrf"  # "rrf" (mới, mặc định) hoặc "weighted" (cách cũ)
ALPHA = 0.82             # chỉ dùng khi FUSION_METHOD == "weighted"
RRF_K = 60               # chỉ dùng khi FUSION_METHOD == "rrf"
HYBRID_TOP_N = 30
NUM_ANSWERS = 5

ENCODE_BATCH_SIZE = 64
RERANK_BATCH_SIZE = 128
QUESTION_CHUNK_SIZE = 100

print("Thiết bị dùng:", GPU_DEVICES)
print("Index sẽ đọc từ:", resolve_index_read_dir())


Thiết bị dùng: ['cuda:0', 'cuda:1']
Index sẽ đọc từ: /kaggle/working/index_cache


## Tách đoạn văn bản

In [5]:
def extract_title(passage, search_end_pos=1000):
    preamble = passage[:search_end_pos]
    pattern = r"(THÔNG TƯ LIÊN TỊCH|THÔNG TƯ|NGHỊ ĐỊNH|QUYẾT ĐỊNH|LUẬT|CÔNG VĂN|HƯỚNG DẪN|QUY CHUẨN)\s*\n+(.*?)(?=Căn cứ|Theo đề nghị|$)"
    match = re.search(pattern, preamble, re.DOTALL)
    if not match:
        return None
    title = match.group(2).replace("\r\n", " ").replace("\n", " ")
    title = re.sub(r"\s+", " ", title).strip()
    return f"{match.group(1)}: {title}"


def split_into_word_windows(text, title, max_words, overlap):
    words = text.split()
    if len(words) <= max_words:
        if title:
            return [f"{title}\n{text.strip()}"]
        return [text.strip()]

    windows = []
    step = max_words - overlap
    for start in range(0, len(words), step):
        window_text = " ".join(words[start:start + max_words])
        if title:
            windows.append(f"{title}\n{window_text}")
        else:
            windows.append(window_text)
    return windows


def chunk_document(passage, doc_id, max_words=MAX_WORDS_PER_CHUNK, overlap=CHUNK_OVERLAP, include_appendix=INCLUDE_APPENDIX):
    article_positions = [match.start() for match in re.finditer(r"Điều\s+\d+\.", passage)]

    if len(article_positions) == 0:
        title = extract_title(passage, search_end_pos=1000)
        windows = split_into_word_windows(passage, title, max_words, overlap)
        chunks = []
        for window_text in windows:
            chunks.append({"doc_id": doc_id, "text": window_text})
        return chunks

    title = extract_title(passage, search_end_pos=article_positions[0])
    appendix_match = re.search(r"PHỤ LỤC", passage)
    body_end = appendix_match.start() if appendix_match else len(passage)
    body = passage[article_positions[0]:body_end]

    article_positions_in_body = [match.start() for match in re.finditer(r"Điều\s+\d+\.", body)]
    article_positions_in_body.append(len(body))

    chunks = []
    for i in range(len(article_positions_in_body) - 1):
        segment = body[article_positions_in_body[i]:article_positions_in_body[i + 1]].strip()
        if len(segment) == 0:
            continue
        # MỚI: nếu 1 Điều dài hơn max_words thì sub-split, tránh bị model cắt cụt khi encode
        # (bản gốc không giới hạn độ dài ở đây, có thể vượt MAX_SEQ_LENGTH)
        if len(segment.split()) > max_words:
            windows = split_into_word_windows(segment, title, max_words, overlap)
        elif title:
            windows = [f"{title}\n{segment}"]
        else:
            windows = [segment]
        for window_text in windows:
            chunks.append({"doc_id": doc_id, "text": window_text})

    if include_appendix and appendix_match:
        appendix_text = passage[appendix_match.start():]
        appendix_positions = [match.start() for match in re.finditer(r"PHỤ LỤC[^\n]{0,40}", appendix_text)]
        appendix_positions.append(len(appendix_text))
        for i in range(len(appendix_positions) - 1):
            segment = appendix_text[appendix_positions[i]:appendix_positions[i + 1]].strip()
            if len(segment) == 0:
                continue
            windows = split_into_word_windows(segment, title, max_words, overlap)
            for window_text in windows:
                chunks.append({"doc_id": doc_id, "text": window_text})

    return chunks


## Xây dựng corpus & chỉ mục

In [6]:
def load_context_records(context_dir):
    records = []
    files = sorted(glob.glob(os.path.join(context_dir, "context_*.json")))
    for file_path in tqdm(files, desc="Loading context files"):
        with open(file_path, encoding="utf-8") as f:
            data = json.load(f)
        if isinstance(data, list):
            records.extend(data)
        else:
            records.append(data)
    return records


def build_corpus(context_dir):
    records = load_context_records(context_dir)
    corpus_texts = []
    corpus_doc_ids = []
    for record in tqdm(records, desc="Chunking documents"):
        doc_chunks = chunk_document(record["passage"], str(record["id"]))
        for chunk in doc_chunks:
            corpus_texts.append(chunk["text"])
            corpus_doc_ids.append(chunk["doc_id"])
    return corpus_texts, corpus_doc_ids


def tokenize_for_bm25(text):
    segmented_text = ViTokenizer.tokenize(str(text))
    cleaned_text = re.sub(r"[^\w\s]", " ", segmented_text)
    return cleaned_text.lower().split()


def build_bm25(corpus_texts):
    tokenized_corpus = []
    for text in tqdm(corpus_texts, desc="Tokenizing for BM25"):
        tokenized_corpus.append(tokenize_for_bm25(text))
    return BM25Okapi(tokenized_corpus)


def encode_corpus(dense_model, corpus_texts):
    if len(GPU_DEVICES) > 1:
        pool = dense_model.start_multi_process_pool(target_devices=GPU_DEVICES)
        embeddings = dense_model.encode_multi_process(corpus_texts, pool, batch_size=ENCODE_BATCH_SIZE)
        dense_model.stop_multi_process_pool(pool)
        embeddings = torch.tensor(embeddings)
    else:
        embeddings = dense_model.encode(
            corpus_texts, batch_size=ENCODE_BATCH_SIZE,
            convert_to_tensor=True, device=PRIMARY_DEVICE,
            show_progress_bar=True,
        )
    return embeddings


def save_index(corpus_texts, corpus_doc_ids, bm25, embeddings):
    with open(os.path.join(INDEX_DIR, "corpus.pkl"), "wb") as f:
        pickle.dump({"texts": corpus_texts, "doc_ids": corpus_doc_ids}, f)
    with open(os.path.join(INDEX_DIR, "bm25.pkl"), "wb") as f:
        pickle.dump(bm25, f)
    torch.save(embeddings, os.path.join(INDEX_DIR, "dense_embeddings.pt"))
    print(f"Đã lưu index vào {INDEX_DIR}")


def load_index_from(read_dir):
    with open(os.path.join(read_dir, "corpus.pkl"), "rb") as f:
        corpus_data = pickle.load(f)
    with open(os.path.join(read_dir, "bm25.pkl"), "rb") as f:
        bm25 = pickle.load(f)
    embeddings = torch.load(os.path.join(read_dir, "dense_embeddings.pt"), weights_only=False)
    return corpus_data["texts"], corpus_data["doc_ids"], bm25, embeddings


def get_or_build_corpus_and_index(dense_model):
    read_dir = resolve_index_read_dir()
    embeddings_file = os.path.join(read_dir, "dense_embeddings.pt")

    if os.path.exists(embeddings_file):
        print(f"Tìm thấy index có sẵn tại {read_dir} — nạp lại.")
        return load_index_from(read_dir)

    print("Không tìm thấy index có sẵn — build mới.")
    corpus_texts, corpus_doc_ids = build_corpus(CONTEXT_DIR)
    bm25 = build_bm25(corpus_texts)
    embeddings = encode_corpus(dense_model, corpus_texts)
    save_index(corpus_texts, corpus_doc_ids, bm25, embeddings)
    return corpus_texts, corpus_doc_ids, bm25, embeddings


In [7]:
dense_model = SentenceTransformer(DENSE_MODEL_NAME, device=PRIMARY_DEVICE)
dense_model.max_seq_length = MAX_SEQ_LENGTH
if PRIMARY_DEVICE.startswith("cuda"):
    dense_model = dense_model.half()

corpus_texts, corpus_doc_ids, bm25, corpus_embeddings = get_or_build_corpus_and_index(dense_model)
corpus_embeddings = corpus_embeddings.to(PRIMARY_DEVICE)

print(f"Số chunk trong corpus: {len(corpus_texts)}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/708 [00:00<?, ?B/s]

...

Loading context files:   0%|          | 0/8532 [00:00<?, ?it/s]

Chunking documents:   0%|          | 0/8532 [00:00<?, ?it/s]

Tokenizing for BM25:   0%|          | 0/488133 [00:00<?, ?it/s]

/tmp/ipykernel_23/69756093.py:42: DeprecationWarning: The `encode_multi_process` method has been deprecated, and its functionality has been integrated into `encode`. You can now call `encode` with the same parameters to achieve multi-process encoding.
  embeddings = dense_model.encode_multi_process(corpus_texts, pool, batch_size=ENCODE_BATCH_SIZE)


Đã lưu index vào /kaggle/working/index_cache
Số chunk trong corpus: 488133


## BM25

In [8]:
def bm25_score(question):
    tokens = tokenize_for_bm25(question)
    return bm25.get_scores(tokens).astype(np.float32)


def bm25_score_batch(questions):
    scores = []
    for question in questions:
        scores.append(bm25_score(question))
    return torch.tensor(np.array(scores), device=PRIMARY_DEVICE)


sample_question = "Việc tổ chức vận động, tiếp nhận nguồn đóng góp tự nguyện được thực hiện dựa trên nguyên tắc nào?"
scores = bm25_score(sample_question)
top_indices = np.argsort(scores)[::-1][:5]
for index in top_indices:
    print(corpus_doc_ids[index], corpus_texts[index][:80])


177504 NGHỊ ĐỊNH: VỀ VẬN ĐỘNG, TIẾP NHẬN, PHÂN PHỐI VÀ SỬ DỤNG CÁC NGUỒN ĐÓNG GÓP TỰ NG
177504 NGHỊ ĐỊNH: VỀ VẬN ĐỘNG, TIẾP NHẬN, PHÂN PHỐI VÀ SỬ DỤNG CÁC NGUỒN ĐÓNG GÓP TỰ NG
177504 NGHỊ ĐỊNH: VỀ VẬN ĐỘNG, TIẾP NHẬN, PHÂN PHỐI VÀ SỬ DỤNG CÁC NGUỒN ĐÓNG GÓP TỰ NG
177504 NGHỊ ĐỊNH: VỀ VẬN ĐỘNG, TIẾP NHẬN, PHÂN PHỐI VÀ SỬ DỤNG CÁC NGUỒN ĐÓNG GÓP TỰ NG
177504 NGHỊ ĐỊNH: VỀ VẬN ĐỘNG, TIẾP NHẬN, PHÂN PHỐI VÀ SỬ DỤNG CÁC NGUỒN ĐÓNG GÓP TỰ NG


## Bi-encoder

In [9]:
def dense_score_batch(questions):
    query_embeddings = dense_model.encode(
        questions, batch_size=ENCODE_BATCH_SIZE,
        convert_to_tensor=True, device=PRIMARY_DEVICE,
    )
    return util.cos_sim(query_embeddings, corpus_embeddings)


dense_scores = dense_score_batch([sample_question])
top_values, top_indices = torch.topk(dense_scores[0], 5)
for index in top_indices.cpu().numpy():
    print(corpus_doc_ids[index], corpus_texts[index][:80])


177504 NGHỊ ĐỊNH: VỀ VẬN ĐỘNG, TIẾP NHẬN, PHÂN PHỐI VÀ SỬ DỤNG CÁC NGUỒN ĐÓNG GÓP TỰ NG
177504 NGHỊ ĐỊNH: VỀ VẬN ĐỘNG, TIẾP NHẬN, PHÂN PHỐI VÀ SỬ DỤNG CÁC NGUỒN ĐÓNG GÓP TỰ NG
139722 Điều 17. Nguyên tắc vận động quyên góp, tiếp nhận tài trợ 1. Việc vận động tài t
177504 NGHỊ ĐỊNH: VỀ VẬN ĐỘNG, TIẾP NHẬN, PHÂN PHỐI VÀ SỬ DỤNG CÁC NGUỒN ĐÓNG GÓP TỰ NG
177504 NGHỊ ĐỊNH: VỀ VẬN ĐỘNG, TIẾP NHẬN, PHÂN PHỐI VÀ SỬ DỤNG CÁC NGUỒN ĐÓNG GÓP TỰ NG


## Kết hợp kết quả

In [10]:
def normalize_scores(score_matrix):
    max_values = score_matrix.max(dim=1, keepdim=True).values
    return score_matrix / (max_values + 1e-9)


def get_score(entry):
    return entry[1]


def dedupe_by_doc_id(chunk_indices, scores, corpus_doc_ids):
    best_score_by_doc = {}
    for chunk_index, score in zip(chunk_indices, scores):
        doc_id = corpus_doc_ids[chunk_index]
        if doc_id not in best_score_by_doc or score > best_score_by_doc[doc_id]:
            best_score_by_doc[doc_id] = score
    return sorted(best_score_by_doc.items(), key=get_score, reverse=True)


def ranks_from_scores(score_matrix):
    """MỚI: đổi ma trận điểm [n_cau_hoi, n_corpus] thành ma trận rank (0 = tốt nhất),
    dùng cho RRF — không phụ thuộc scale điểm giữa BM25 và cosine similarity."""
    order = torch.argsort(score_matrix, dim=1, descending=True)
    ranks = torch.empty_like(order)
    arange = torch.arange(score_matrix.shape[1], device=score_matrix.device).expand_as(order)
    ranks.scatter_(1, order, arange)
    return ranks


def hybrid_candidates(questions, alpha=ALPHA, top_n=HYBRID_TOP_N,
                       fusion_method=FUSION_METHOD, rrf_k=RRF_K):
    bm25_scores = bm25_score_batch(questions)
    dense_scores = dense_score_batch(questions)

    if fusion_method == "rrf":
        # MỚI: Reciprocal Rank Fusion — chỉ dùng thứ hạng, không dùng điểm thô,
        # nên không bị lệch bởi thang điểm khác nhau giữa BM25 và dense.
        bm25_ranks = ranks_from_scores(bm25_scores)
        dense_ranks = ranks_from_scores(dense_scores)
        final_scores = 1.0 / (rrf_k + bm25_ranks.float() + 1) + 1.0 / (rrf_k + dense_ranks.float() + 1)
    else:
        bm25_normalized = normalize_scores(bm25_scores)
        dense_normalized = normalize_scores(dense_scores)
        final_scores = alpha * dense_normalized + (1 - alpha) * bm25_normalized

    top_values, top_indices = torch.topk(final_scores, top_n, dim=1)
    return top_indices.cpu().numpy(), top_values.cpu().numpy()


top_indices, top_values = hybrid_candidates([sample_question])
ranked_docs = dedupe_by_doc_id(top_indices[0], top_values[0], corpus_doc_ids)
for doc_id, score in ranked_docs[:5]:
    print(f"{score:.4f}", doc_id)


0.0325 177504
0.0257 287259
0.0240 101100
0.0234 139722
0.0224 218452


## Hiệu chỉnh trọng số Hybrid

In [11]:
def precision_recall(predicted_doc_ids, gold_doc_ids, max_answers=NUM_ANSWERS):
    predicted_set = set(predicted_doc_ids)
    if len(predicted_set) == 0 or len(predicted_set) > max_answers:
        return 0.0, 0.0
    true_positives = len(predicted_set & gold_doc_ids)
    precision = true_positives / len(predicted_set)
    recall = true_positives / len(gold_doc_ids) if len(gold_doc_ids) > 0 else 0.0
    return precision, recall


def search_best_alpha(questions, golds, alpha_candidates, top_n=HYBRID_TOP_N, chunk_size=QUESTION_CHUNK_SIZE):
    print(f"{'Alpha':<8}{'Precision':>12}{'Recall':>10}")
    best_alpha, best_recall, best_precision = None, -1.0, None

    for alpha in alpha_candidates:
        precision_sum = 0.0
        recall_sum = 0.0
        question_count = 0

        for start in range(0, len(questions), chunk_size):
            end = min(start + chunk_size, len(questions))
            batch_questions = questions[start:end]
            batch_golds = golds[start:end]

            top_indices, top_values = hybrid_candidates(batch_questions, alpha=alpha, top_n=top_n)
            for i in range(len(batch_questions)):
                ranked_docs = dedupe_by_doc_id(top_indices[i], top_values[i], corpus_doc_ids)
                predicted_doc_ids = []
                for doc_id, score in ranked_docs[:NUM_ANSWERS]:
                    predicted_doc_ids.append(doc_id)
                precision, recall = precision_recall(predicted_doc_ids, batch_golds[i])
                precision_sum += precision
                recall_sum += recall
                question_count += 1

        average_precision = precision_sum / question_count
        average_recall = recall_sum / question_count
        print(f"{alpha:<8}{average_precision:>12.4f}{average_recall:>10.4f}")

        if average_recall > best_recall:
            best_recall, best_alpha, best_precision = average_recall, alpha, average_precision

    print(f"\nAlpha tối ưu Recall: {best_alpha} (Recall={best_recall:.4f}, Precision={best_precision:.4f})")
    return best_alpha


with open(TRAIN_FILE, encoding="utf-8") as f:
    train_data = json.load(f)

random.seed(42)
sample_size = min(800, len(train_data))
sample_qids = random.sample(list(train_data.keys()), sample_size)

sample_questions = []
sample_golds = []
for qid in sample_qids:
    sample_questions.append(train_data[qid]["question"])
    sample_golds.append(set(train_data[qid]["answer"]))

# Lưu ý: alpha chỉ có tác dụng khi FUSION_METHOD == "weighted". Nếu đang dùng "rrf" (mặc định), bước hiệu chỉnh alpha bên dưới không ảnh hưởng kết quả cuối.
# Bỏ comment dòng dưới để hiệu chỉnh lại alpha trên dữ liệu train (chỉ cần nếu bạn đổi FUSION_METHOD = "weighted").
# ALPHA = search_best_alpha(sample_questions, sample_golds, [0.5, 0.6, 0.7, 0.75, 0.8, 0.82, 0.85, 0.9])
print("FUSION_METHOD đang dùng:", FUSION_METHOD, "| ALPHA:", ALPHA, "| RRF_K:", RRF_K)


FUSION_METHOD đang dùng: rrf | ALPHA: 0.82 | RRF_K: 60


## Cross-encoder

In [12]:
reranker_devices = GPU_DEVICES[:2] if len(GPU_DEVICES) > 1 else [PRIMARY_DEVICE]

rerankers = []
for device in reranker_devices:
    rerankers.append(CrossEncoder(RERANKER_MODEL_NAME, max_length=RERANKER_MAX_LENGTH, device=device))

print("Reranker chạy trên:", reranker_devices)


def rerank_pairs(pairs):
    if len(rerankers) == 1:
        return rerankers[0].predict(pairs, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)

    midpoint = len(pairs) // 2
    first_half = pairs[:midpoint]
    second_half = pairs[midpoint:]

    with ThreadPoolExecutor(max_workers=2) as executor:
        first_future = executor.submit(rerankers[0].predict, first_half, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)
        second_future = executor.submit(rerankers[1].predict, second_half, batch_size=RERANK_BATCH_SIZE, show_progress_bar=False)
        first_scores = first_future.result()
        second_scores = second_future.result()

    return np.concatenate([first_scores, second_scores])


sample_pairs = []
for index in top_indices[0]:
    sample_pairs.append([sample_question, corpus_texts[index]])

sample_rerank_scores = rerank_pairs(sample_pairs)
ranked_docs = dedupe_by_doc_id(top_indices[0], sample_rerank_scores, corpus_doc_ids)
for doc_id, score in ranked_docs[:5]:
    print(f"{score:.4f}", doc_id)


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranker chạy trên: ['cuda:0', 'cuda:1']
1.0000 177504
0.7420 139722
0.6945 66264
0.6723 287259
0.5870 218452


## Chạy pipeline & Lưu kết quả

In [13]:
def load_questions(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def load_checkpoint(path):
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        print(f"Checkpoint có sẵn: {len(data)} câu đã có kết quả.")
        return data
    return {}


def save_checkpoint(submission, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(submission, f, ensure_ascii=False, indent=2)


def run_pipeline(questions_data, checkpoint_path):
    submission = load_checkpoint(checkpoint_path)

    remaining_qids = []
    for qid in questions_data:
        if qid not in submission:
            remaining_qids.append(qid)

    print(f"Còn {len(remaining_qids)}/{len(questions_data)} câu hỏi cần chạy.")

    for start in tqdm(range(0, len(remaining_qids), QUESTION_CHUNK_SIZE), desc="Running pipeline"):
        end = min(start + QUESTION_CHUNK_SIZE, len(remaining_qids))
        batch_qids = remaining_qids[start:end]

        batch_questions = []
        for qid in batch_qids:
            batch_questions.append(questions_data[qid]["question"])

        top_indices, top_values = hybrid_candidates(batch_questions, alpha=ALPHA, top_n=HYBRID_TOP_N)

        all_pairs = []
        pair_counts = []
        for i, question in enumerate(batch_questions):
            count = 0
            for chunk_index in top_indices[i]:
                all_pairs.append([question, corpus_texts[chunk_index]])
                count += 1
            pair_counts.append(count)

        rerank_scores = rerank_pairs(all_pairs)

        cursor = 0
        for i, qid in enumerate(batch_qids):
            count = pair_counts[i]
            scores_i = rerank_scores[cursor:cursor + count]
            cursor += count

            ranked_docs = dedupe_by_doc_id(top_indices[i], scores_i, corpus_doc_ids)
            answer_doc_ids = []
            for doc_id, score in ranked_docs[:NUM_ANSWERS]:
                answer_doc_ids.append(doc_id)
            submission[qid] = {"answer": answer_doc_ids}

        save_checkpoint(submission, checkpoint_path)
        torch.cuda.empty_cache()
        gc.collect()

    return submission


In [14]:
INPUT_FILE = PUBLIC_TEST_FILE

questions_data = load_questions(INPUT_FILE)
submission = run_pipeline(questions_data, CHECKPOINT_FILE)

for qid, item in submission.items():
    if len(item["answer"]) == 0 or len(item["answer"]) > NUM_ANSWERS:
        raise ValueError(f"Câu {qid} có số lượng đáp án không hợp lệ: {len(item['answer'])}")

with open(SUBMISSION_FILE, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)

print(f"Đã ghi {len(submission)} câu trả lời vào {SUBMISSION_FILE}")


Còn 1000/1000 câu hỏi cần chạy.


Running pipeline:   0%|          | 0/10 [00:00<?, ?it/s]

Đã ghi 1000 câu trả lời vào /kaggle/working/outputs/submission.json


## Chấm điểm

In [15]:
def eval_retrieval(y_pred, y_true):
    predicted_answers = {}
    for qid, item in y_pred.items():
        predicted_answers[qid] = item["answer"]

    prediction_ids = list(predicted_answers.keys())
    truth_ids = list(y_true.keys())

    if len(prediction_ids) != len(truth_ids):
        raise Exception("Samples in predict not match with reference")

    recall_scores = []
    for qid in truth_ids:
        predicted = predicted_answers.get(qid, set())
        if len(predicted) > 0 and len(predicted) <= 5:
            correct = set(y_true[qid]) & set(predicted)
            recall_scores.append(len(correct) / len(y_true[qid]))
        else:
            recall_scores.append(0)

    precision_scores = []
    for qid in prediction_ids:
        predicted = predicted_answers[qid]
        if len(predicted) > 0 and len(predicted) <= 5:
            correct = set(y_true[qid]) & set(predicted)
            precision_scores.append(len(correct) / len(predicted))
        else:
            precision_scores.append(0)

    recall = sum(recall_scores) / len(recall_scores)
    precision = sum(precision_scores) / len(precision_scores)
    return {"precision": precision, "recall": recall}


def score_against_train(submission, train_file=TRAIN_FILE):
    with open(train_file, encoding="utf-8") as f:
        train_data = json.load(f)

    ground_truth = {}
    for qid, item in train_data.items():
        ground_truth[qid] = item["answer"]

    scores = eval_retrieval(submission, ground_truth)
    print("Final scores:", scores)

    with open(os.path.join(OUTPUT_DIR, "scores.json"), "w") as f:
        json.dump(scores, f)

    return scores


# Chỉ chạy được nếu submission ở trên được sinh ra từ TRAIN_FILE (đổi INPUT_FILE rồi chạy lại phần trước).
# scores = score_against_train(submission, TRAIN_FILE)


## Tóm tắt pipeline

1. **Tách đoạn văn bản** — chia mỗi văn bản luật thành các đoạn theo từng "Điều", kèm tiêu đề và phụ lục, thay vì giữ nguyên cả văn bản dài.
2. **Xây dựng corpus & chỉ mục** — build BM25 index và embedding (Bi-encoder, mã hóa song song trên 2 GPU) cho toàn bộ đoạn văn bản. Có checkpoint, chỉ build khi chưa có sẵn.
3. **BM25** — chấm điểm theo từ khóa, dùng `pyvi` tách từ tiếng Việt.
4. **Bi-encoder** — chấm điểm theo ngữ nghĩa bằng model tiếng Việt `AITeamVN/Vietnamese_Embedding`.
5. **Kết hợp kết quả** — chuẩn hóa và cộng có trọng số 2 loại điểm (`alpha * dense + (1-alpha) * bm25`), gộp các đoạn cùng văn bản lại, giữ điểm cao nhất.
6. **Hiệu chỉnh trọng số Hybrid** — tìm `alpha` tối ưu Recall trên mẫu từ `train.json` (mặc định dùng giá trị đã hiệu chỉnh sẵn).
7. **Cross-encoder** — chấm điểm lại bằng model tiếng Việt `AITeamVN/Vietnamese_Reranker`, chạy song song trên 2 GPU.
8. **Chạy pipeline & Lưu kết quả** — luôn cắt lấy đúng 5 văn bản/câu hỏi (theo xác nhận Recall là tiêu chí chính của BTC), tự lưu checkpoint để resume nếu bị ngắt.
9. **Chấm điểm** — tự đánh giá Recall/Precision trên `train.json` bằng đúng công thức BTC dùng.

## Thay đổi so với bản gốc

- **Chunking**: đoạn theo "Điều" nếu dài hơn `MAX_WORDS_PER_CHUNK` giờ được sub-split thành nhiều chunk nhỏ (bản gốc không giới hạn, có thể vượt `MAX_SEQ_LENGTH` và bị model cắt cụt khi encode).
- **Fusion**: thêm tuỳ chọn **RRF** (`FUSION_METHOD = "rrf"`, mặc định) bên cạnh cách cộng điểm có trọng số cũ (`"weighted"`) — RRF dùng thứ hạng thay vì điểm thô nên không bị lệch bởi khác thang đo giữa BM25 và cosine similarity. Muốn quay lại cách cũ, đổi `FUSION_METHOD = "weighted"` ở phần Cấu hình.
